#### **If the punc_model is not downloaded yet then download it here**

In [19]:
# import os
# import shutil
# import sys
# from huggingface_hub import snapshot_download


# cache_dir = "./capu"
# sys.path.append(cache_dir)


# def download_files(repo_id, cache_dir=None, ignore_patterns=None):
#     download_dir = snapshot_download(repo_id=repo_id, cache_dir=cache_dir, ignore_patterns=ignore_patterns)
#     if cache_dir is None or download_dir == cache_dir:
#         return download_dir
#     file_names = os.listdir(download_dir)
#     for file_name in file_names:
#         shutil.move(os.path.join(download_dir, file_name), cache_dir)
#     os.rmdir(download_dir)
#     return cache_dir


# cache_dir = download_files(repo_id="dragonSwing/xlm-roberta-capu", cache_dir=cache_dir)        

#### **Model for punctuations**

In [1]:
cache_dir = "./capu"
import os
import sys
sys.path.append(cache_dir)

from gec_model import GecBERTModel
punc_model = GecBERTModel(
    vocab_path=os.path.join(cache_dir, "vocabulary"),
    model_paths="dragonSwing/xlm-roberta-capu",
    split_chunk=True
)

print(punc_model)

C:\Users\PC\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\PC\miniconda3\envs\aic-sub\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


GecBERTModel()


#### **Model for text embedding**

In [2]:
from sentence_transformers import SentenceTransformer
ebd_model = SentenceTransformer('keepitreal/vietnamese-sbert')

c:\Users\PC\miniconda3\envs\aic-sub\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [7]:
import json
import re
from youtube_transcript_api import YouTubeTranscriptApi
from sklearn.metrics.pairwise import cosine_similarity
from preprocess import preprocessing
from feature_extraction import get_svd_vectors, get_ebd_vectors


METADATA_DIR_PATH = "E:/2023 HCM AI CHALLENGE/media-info"
TRANSCRIPT_DIR_PATH = "E:/2023 HCM AI CHALLENGE/transcripts"
TRANSCRIPT_SPLIT_DIR_PATH = "E:/2023 HCM AI CHALLENGE/transcripts_split"


def get_video_web_id(video_id: str) -> str:
    with open(f"{METADATA_DIR_PATH}/{video_id}.json", "r", encoding="utf-8") as f:
        video_metadata = json.load(f)
        try:
            video_web_id = video_metadata["watch_url"].split("=")[-1]
        except:
            video_web_id = None
    return video_web_id


def get_transcript(video_web_id: str) -> str:
    srt = YouTubeTranscriptApi.get_transcript(video_web_id, languages=['vi'])
    texts = [srt[i]["text"] for i in range(len(srt))]
    transcript = ' '.join(texts)
    return transcript


def get_corrected_transcript(transcript: str) -> str:
    corrected_transcript = punc_model(transcript)[0]
    return corrected_transcript

In [10]:
# main

video_ids = ["L07_V001", "L07_V002"]
for video_id in video_ids:
    corrected_transcript = "" 

    # if transcript is not generated -> generate it
    if f"{video_id}.txt" not in os.listdir(TRANSCRIPT_DIR_PATH):
        video_web_id = get_video_web_id(video_id)
        if video_web_id is None:
            continue
        transcript = get_transcript(video_web_id)
        corrected_transcript = get_corrected_transcript(transcript)
        with open(f"{TRANSCRIPT_DIR_PATH}/{video_id}.txt", "w", encoding="utf-16") as f:
            f.write(corrected_transcript)
    # if transcript is generated -> read it
    else:
        with open(f"{TRANSCRIPT_DIR_PATH}/{video_id}.txt", "r", encoding="utf-16") as f:
            corrected_transcript = f.read()

    print()
    print(f"{video_id}: {corrected_transcript[:100]}...{corrected_transcript[-100:]}")

    # split transcript into documents and preprocess them
    documents = []
    with open(f"{TRANSCRIPT_DIR_PATH}/{video_id}.txt", "r", encoding="utf-16") as file:
        documents = re.split(r'(?<!\b\w)\.\s+|(?<!\b\w)\:\s+', file.read())
    print("Number of documents: ", len(documents))
    preprocessed_documents = [preprocessing(doc) for doc in documents]

    # get vectors
    svd_vectors = get_svd_vectors(preprocessed_documents)
    ebd_vectors = get_ebd_vectors(preprocessed_documents, ebd_model)

    HIGH_EBD_THRESHOLD = 0.4
    LOW_EBD_THRESHOLD = 0.1
    SVD_THRESHOLD = 0.2
    MIN_DOC_LENGTH = 100

    svd_similarities = [cosine_similarity(svd_vectors[i].reshape((1, -1)), svd_vectors[i + 1].reshape((1, -1))) for i in range(len(preprocessed_documents) - 1)]
    embedding_similarities = [cosine_similarity(ebd_vectors[i].reshape((1, -1)), ebd_vectors[i + 1].reshape((1, -1))) for i in range(len(preprocessed_documents) - 1)]
    svd_similarities.insert(0, 0)
    embedding_similarities.insert(0, 0)

    contexts = []
    i = 0
    while i < len(documents):
        context = {
            "start_sentence_id": i,
            "end_sentence_id": i - 1,
            "context_length": 0,
        }
        for j in range(i + 1, len(documents) + 1):
            is_new_context = False
            if j == len(documents):
                is_new_context = True
            elif len(documents[j]) < MIN_DOC_LENGTH:
                is_new_context = False
            elif documents[j].lower().startswith("thưa quý vị"):
                is_new_context = True
            elif embedding_similarities[j] > HIGH_EBD_THRESHOLD:
                is_new_context = False
            elif embedding_similarities[j] < LOW_EBD_THRESHOLD:
                is_new_context = True
            elif svd_similarities[j] > SVD_THRESHOLD:
                is_new_context = False
            else:
                is_new_context = True

            context["end_sentence_id"] = j - 1
            context["context_length"] += len(documents[j - 1])
            if is_new_context: 
                contexts.append(context)           
                i = j
                break

    with open(f"{TRANSCRIPT_SPLIT_DIR_PATH}/{video_id}.txt", "w", encoding="utf-16") as f:
        for context in contexts:
            start = context["start_sentence_id"]
            end = context["end_sentence_id"]
            f.write("\n")
            f.write(f"Context length: {context['context_length']}\n")
            for i in range(start, end + 1):
                # print(f"{embedding_similarities[i]}\t{svd_similarities[i]}\t{documents[i]}")
                f.write(f"{documents[i]}\n")

    

    

    


L07_V001: Chào mừng quý vị đến với chương trình 60 giây của Đài truyền hình thành phố Hồ Chí Minh. Chương trìn...eo dõi. Hẹn gặp lại quý vị trong Chương Trình 60 Giây Chiều Nay. Còn Bây Giờ Xin Kính Chào Tạm Biệt.
Number of documents:  114
Tf-idf shape: (114, 552)
(114, 768)

L07_V002: Kính chào quý vị, đây là chương trình 60 Giây Sáng của Đài truyền hình thành phố Hồ Chí Minh mang đế... lại quý vị vào chương trình 60 Giây chiều nay với những thông tin cập nhật. Xin kính chào tạm biệt.
Number of documents:  124
Tf-idf shape: (124, 577)
(124, 768)


## Spacy english

In [ ]:
import numpy as np
import spacy

# Load the Spacy model
nlp = spacy.load('en_core_web_sm')


text = text3[0]


def process(text):
    doc = nlp(text)
    sents = list(doc.sents)
    vecs = np.stack([sent.vector / sent.vector_norm for sent in sents])

    return sents, vecs

def cluster_text(sents, vecs, threshold):
    clusters = [[0]]
    for i in range(1, len(sents)):
        if np.dot(vecs[i], vecs[i-1]) < threshold:
            clusters.append([])
        clusters[-1].append(i)
    
    return clusters

def clean_text(text):
    # Add your text cleaning process here
    return text

# Initialize the clusters lengths list and final texts list
clusters_lens = []
final_texts = []

# Process the chunk
threshold = 0.3
sents, vecs = process(text)

# Cluster the sentences
clusters = cluster_text(sents, vecs, threshold)

for cluster in clusters:
    cluster_txt = clean_text(' '.join([sents[i].text for i in cluster]))
    cluster_len = len(cluster_txt)
    
    # Check if the cluster is too short
    if cluster_len < 60:
        continue
    
    # Check if the cluster is too long
    elif cluster_len > 3000:
        threshold = 0.6
        sents_div, vecs_div = process(cluster_txt)
        reclusters = cluster_text(sents_div, vecs_div, threshold)
        
        for subcluster in reclusters:
            div_txt = clean_text(' '.join([sents_div[i].text for i in subcluster]))
            div_len = len(div_txt)
            
            if div_len < 60 or div_len > 3000:
                continue
            
            clusters_lens.append(div_len)
            final_texts.append(div_txt)
            
    else:
        clusters_lens.append(cluster_len)
        final_texts.append(cluster_txt)